# Week 3: Local LLMs with Ollama — Privacy-First AI

Welcome to Week 3. Last week you built a semantic search index over pension documents. This week you add the intelligence layer: a local large language model that runs entirely on your machine.

**Learning goals:**
- Understand why local LLMs matter for pension-domain work
- Install Ollama and pull open-weight models
- Make your first generation call and understand the response structure
- Explore temperature, system prompts, and context windows
- Hit the context-window wall — motivating RAG in Week 5

## 1. Why Local LLMs?

When you work with pension fund data you face constraints that make cloud LLM APIs risky or impractical:

| Concern | Cloud API | Local Ollama |
|---|---|---|
| **Privacy** | Data leaves your network | Stays on your machine |
| **Cost during learning** | Pay per token | Free after hardware |
| **Offline capability** | Requires internet | Works air-gapped |
| **Reproducibility** | Model versions change | You pin the version |
| **Regulatory risk** | Data sent to US servers | Zero data egress |

Pension data often includes beneficiary counts, coverage ratios, and fund-specific financials that are non-public. Sending these through a third-party API creates compliance risk under GDPR and Dutch pension regulations.

### Setup Instructions

1. Install Ollama from [ollama.com](https://ollama.com)
2. Start the server: `ollama serve` (or it auto-starts on macOS/Linux)
3. Pull the models we'll use:
```bash
ollama pull llama3.2
ollama pull mistral
```
4. Verify: `ollama list` should show both models

**Model sizes:**
- `llama3.2` (3B params) — ~2 GB RAM, fast on CPU, great for learning
- `mistral` (7B params) — ~4.5 GB RAM, higher quality, still CPU-runnable
- `llama3.2:latest` defaults to 3B; use `llama3.2:1b` for the smallest footprint

Context windows: llama3.2 = 128k tokens, mistral = 32k tokens

## 2. Checking Ollama is Running

In [ ]:
import requests
import json

OLLAMA_BASE = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    response.raise_for_status()
    data = response.json()
    models = data.get("models", [])
    print(f"Ollama is running. Found {len(models)} model(s):\n")
    for m in models:
        size_gb = m.get('size', 0) / 1e9
        print(f"  {m['name']:40s}  {size_gb:.1f} GB")
except requests.ConnectionError:
    print("ERROR: Cannot connect to Ollama.")
    print("Start it with: ollama serve")
    print("Or on macOS/Linux it may start automatically — check Activity Monitor/ps aux")
except Exception as e:
    print(f"Unexpected error: {e}")

## 3. First Generation — Anatomy of a Prompt

LLMs work with a sequence of **messages**, each tagged with a **role**:

| Role | Purpose | Example |
|---|---|---|
| `system` | Sets model behavior, persona, constraints | "You are a pension regulation expert" |
| `user` | The human's input | "What is a coverage ratio?" |
| `assistant` | The model's previous replies | "A coverage ratio is..." |

The model sees the **entire message history** on each call — there is no persistent memory on the server. You are responsible for managing conversation state.

The `ollama` Python library wraps the HTTP API with a clean interface.

In [ ]:
import ollama

MODEL = "llama3.2"  # Change to "mistral" if you prefer

messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant specializing in Dutch pension regulations."
    },
    {
        "role": "user",
        "content": "In one sentence, what is a coverage ratio in the context of pension funds?"
    }
]

response = ollama.chat(model=MODEL, messages=messages)

print("=== Response object keys ===")
print(list(response.keys()) if hasattr(response, 'keys') else dir(response))
print()
print("=== Answer ===")
print(response['message']['content'])
print()
print("=== Usage ===")
print(f"Prompt tokens:    {response.get('prompt_eval_count', 'N/A')}")
print(f"Response tokens:  {response.get('eval_count', 'N/A')}")
print(f"Total duration:   {response.get('total_duration', 0) / 1e9:.2f}s")

## 4. Streaming Responses

Without streaming, your application blocks until the full response is ready — which can be 10–30 seconds for longer answers. Users see nothing, then a wall of text.

With streaming, tokens arrive as they are generated. This creates the familiar "typing" effect and makes the application feel responsive. For a production chatbot, streaming is almost always the right choice.

Ollama returns a stream of newline-delimited JSON objects. Each has a `message` field with the partial content, and a final object with `done: true`.

In [ ]:
import ollama
import sys

question = "Explain the three funding regimes in the Dutch FTK (Financieel Toetsingskader) in 3 bullet points."

messages = [
    {"role": "system", "content": "You are a pension regulation expert. Be concise and accurate."},
    {"role": "user", "content": question}
]

print(f"Question: {question}\n")
print("Answer (streaming):\n")

full_response = ""
for chunk in ollama.chat(model=MODEL, messages=messages, stream=True):
    token = chunk['message']['content']
    full_response += token
    print(token, end="", flush=True)  # flush=True is critical for real-time display

print("\n\n--- Stream complete ---")
print(f"Total characters received: {len(full_response)}")

## 5. Token Counting

**What is a token?** A token is roughly 3/4 of a word in English. "pension" = 1 token, "unfunded liability" = 3 tokens. Special characters, numbers, and rare words often become multiple tokens.

Why does this matter?
- Models have a **context window** limit — the maximum tokens they can process at once
- `llama3.2`: 128,000 tokens (~96,000 words — about a full novel)
- `mistral`: 32,000 tokens (~24,000 words — about a long report)
- Costs (for cloud APIs) are billed per token
- **Latency** grows roughly linearly with token count

We use `tiktoken` (OpenAI's tokenizer) as a fast approximation. The exact tokenizer varies by model but token counts are within ~10%.

In [ ]:
try:
    import tiktoken
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tiktoken", "-q"])
    import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # Used by GPT-4, close enough for estimation

# Simulate a typical pension regulation abstract
article_abstract = """
This paper examines the impact of the revised Financieel Toetsingskader (FTK) on Dutch occupational
pension funds' asset allocation strategies. Using a dataset of 187 pension funds from 2015 to 2023,
we find that funds subject to the enhanced recovery plan requirements systematically reduced equity
exposure by 8.3 percentage points on average (p<0.01), shifting toward liability-driven investment
strategies. The coverage ratio threshold of 104.2% introduces a discontinuity in investment behavior,
creating a "cliff effect" where funds near the threshold adopt markedly more conservative portfolios.
We argue that this regulatory architecture, while prudent from a solvency perspective, may generate
suboptimal long-term returns for beneficiaries and recommends smoothing mechanisms similar to those
adopted in the 2023 Wet toekomst pensioenen reform.
"""

tokens = enc.encode(article_abstract)
print(f"Abstract length: {len(article_abstract)} characters")
print(f"Token count:     {len(tokens)} tokens")
print(f"Chars per token: {len(article_abstract)/len(tokens):.1f}")
print()

# How many articles can fit in context?
llama32_context = 128_000
mistral_context = 32_000
system_prompt_overhead = 200

print("Context window capacity:")
for model_name, ctx in [("llama3.2 (128k)", llama32_context), ("mistral (32k)", mistral_context)]:
    available = ctx - system_prompt_overhead
    articles_fit = available // len(tokens)
    print(f"  {model_name}: {articles_fit} abstracts of this length")

print()
print("First 20 tokens decoded:")
for i, tok_id in enumerate(tokens[:20]):
    print(f"  [{tok_id:6d}] '{enc.decode([tok_id])}'")

## 6. Temperature Experiments

**Temperature** controls the randomness of the model's outputs:

- **0.0** — Deterministic (or nearly so). Always picks the highest-probability token. Best for factual Q&A, data extraction, classification.
- **0.5** — Balanced. Some creativity, still mostly coherent. Good for summarization, drafting.
- **1.0** — High creativity. More varied vocabulary, different angles on the same question. Best for brainstorming, creative writing. Can hallucinate more.

For pension-domain work where accuracy matters, you almost always want **temperature = 0.0 or 0.1**.

In [ ]:
import ollama

question = "What are the main risks facing defined-benefit pension funds in the Netherlands?"

temperatures = [0.0, 0.5, 1.0]
results = []

print(f"Question: {question}\n")
print("Generating responses at different temperatures...\n")

for temp in temperatures:
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a pension risk analyst. Answer in exactly 2 sentences."},
            {"role": "user", "content": question}
        ],
        options={"temperature": temp, "seed": 42}
    )
    answer = response['message']['content'].strip()
    results.append((temp, answer))
    print(f"Temperature {temp}: {answer[:100]}..." if len(answer) > 100 else f"Temperature {temp}: {answer}")
    print()

# Display as comparison table
print("\n" + "="*60)
print("COMPARISON TABLE")
print("="*60)
for temp, answer in results:
    print(f"\nTemp={temp}:")
    print(f"  {answer}")
    word_count = len(answer.split())
    print(f"  [word count: {word_count}]")

print("\n" + "="*60)
print("GUIDANCE:")
print("  temp=0.0 → Use for: data extraction, fact lookup, classification")
print("  temp=0.5 → Use for: summarization, report drafting")
print("  temp=1.0 → Use for: brainstorming, generating multiple framings")

## 7. System Prompts and Personas

The system prompt is the most powerful tool in prompt engineering. It sets:
- **Role/persona**: Who is the model pretending to be?
- **Tone**: Formal, casual, technical, explanatory?
- **Constraints**: Only answer about X, refuse to discuss Y
- **Format**: Always respond in JSON, use bullet points, max 3 sentences

A well-crafted system prompt can make the same 3B model behave like a specialist.

In [ ]:
import ollama

question = "Should I move my pension fund assets to more equities given the current interest rate environment?"

personas = [
    (
        "Generic assistant",
        "You are a helpful assistant."
    ),
    (
        "Pension regulation expert",
        """You are a senior pension regulation expert with 20 years of experience advising Dutch
        occupational pension funds (bedrijfstakpensioenfondsen). You speak precisely, reference
        regulatory frameworks like the FTK and Wet toekomst pensioenen where relevant, and always
        note when advice depends on a fund's specific coverage ratio (dekkingsgraad) and investment
        horizon. You do not give personal financial advice — you explain regulatory and actuarial
        principles."""
    ),
    (
        "Explain like I'm 5",
        "You are a teacher explaining complex finance concepts to a 10-year-old. Use simple words, "
        "analogies, and short sentences. Avoid jargon entirely."
    )
]

print(f"Question: {question}\n")
print("="*60)

for persona_name, system_prompt in personas:
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ],
        options={"temperature": 0.1}
    )
    print(f"\n--- Persona: {persona_name} ---")
    print(response['message']['content'][:400])
    print("..." if len(response['message']['content']) > 400 else "")

print("\n" + "="*60)
print("Notice how the same model produces radically different tone and content.")
print("The expert persona references specific Dutch regulations.")
print("The ELI5 persona uses simple analogies.")

## 8. The Context Window Problem

Now let's hit the wall that motivates RAG.

You have ~1,000+ articles in your Phase 1 dataset. Even with llama3.2's 128k context window, you cannot stuff all article text into one prompt:
- 1,000 articles × ~200 tokens/abstract = 200,000 tokens — exceeds llama3.2's limit
- Even if it fit, performance degrades with very long contexts ("lost in the middle" problem)
- You'd pay (in latency) for all that context even if only 3 articles are relevant

**The solution is RAG**: instead of stuffing everything in, retrieve only the relevant pieces.

In [ ]:
import pandas as pd
import ollama

try:
    import tiktoken
    enc = tiktoken.get_encoding("cl100k_base")
except ImportError:
    enc = None

articles_path = "../data/raw/articles.csv"

try:
    df = pd.read_csv(articles_path)
    print(f"Loaded {len(df)} articles from {articles_path}")
    print(f"Columns: {list(df.columns)}")

    abstract_col = "abstract" if "abstract" in df.columns else df.columns[1]
    sample = df.head(10)

    combined = "\n\n".join(
        f"Article {i+1}: {row[abstract_col]}"
        for i, (_, row) in enumerate(sample.iterrows())
        if pd.notna(row.get(abstract_col, ""))
    )

    if enc:
        token_count = len(enc.encode(combined))
        print(f"\n10 articles combined: {len(combined)} chars, ~{token_count} tokens")

        full_df_abstracts = " ".join(df[abstract_col].dropna().astype(str))
        full_tokens = len(enc.encode(full_df_abstracts))
        print(f"All {len(df)} abstracts: ~{full_tokens:,} tokens")
        print(f"  llama3.2 limit: 128,000 tokens — {'FITS' if full_tokens < 128000 else 'TOO LARGE'}")
        print(f"  mistral limit:   32,000 tokens — {'FITS' if full_tokens < 32000 else 'TOO LARGE'}")

    # Attempt generation with 10-article context
    prompt = f"Based on these 10 articles, what are the main themes?\n\n{combined[:3000]}"
    print("\nCalling model with combined context (first 3000 chars shown)...")
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.0}
    )
    print("Response:", response['message']['content'][:300])
    print("\n=> This 'works' with 10 articles, but imagine 1000...")
    print("=> RAG (Week 5) solves this by retrieving ONLY the relevant chunks.")

except FileNotFoundError:
    print(f"File not found: {articles_path}")
    print("This is expected if you haven't run Phase 1 yet.")
    print("The key insight: even 1000 short abstracts overflow most context windows.")
    print("Token math: 1000 articles × 150 tokens = 150,000 tokens > llama3.2's 128k limit.")

## 9. Calling Ollama: Library vs Raw HTTP

The `ollama` Python library is a thin wrapper. Understanding the raw HTTP API helps when:
- You're debugging network issues
- You're building in a language without an official SDK
- You need fine-grained control over request parameters

Both approaches are equivalent — use whichever feels cleaner.

In [ ]:
import ollama
import requests
import json

question = "What does 'dekkingsgraad' mean in Dutch pension law?"
messages = [{"role": "user", "content": question}]

# ── Method 1: ollama Python library ──────────────────────────────────────────
print("METHOD 1: ollama library")
resp_lib = ollama.chat(
    model=MODEL,
    messages=messages,
    options={"temperature": 0.0}
)
answer_lib = resp_lib['message']['content']
print(f"Answer: {answer_lib[:200]}")
print()

# ── Method 2: raw requests.post ──────────────────────────────────────────────
print("METHOD 2: raw HTTP POST to /api/chat")
payload = {
    "model": MODEL,
    "messages": messages,
    "stream": False,
    "options": {"temperature": 0.0}
}
resp_http = requests.post(
    f"{OLLAMA_BASE}/api/chat",
    json=payload,
    timeout=60
)
data_http = resp_http.json()
answer_http = data_http['message']['content']
print(f"Answer: {answer_http[:200]}")
print()

# ── Verify they're equivalent ─────────────────────────────────────────────────
print("Both methods produced similar-length responses:")
print(f"  Library:   {len(answer_lib)} chars")
print(f"  Raw HTTP:  {len(answer_http)} chars")
print()
print("The library is cleaner; raw HTTP is useful for debugging and non-Python integrations.")
print(f"\nOther useful Ollama endpoints:")
print(f"  GET  {OLLAMA_BASE}/api/tags          — list models")
print(f"  POST {OLLAMA_BASE}/api/generate      — single-turn, no messages")
print(f"  POST {OLLAMA_BASE}/api/embeddings    — get embeddings")
print(f"  POST {OLLAMA_BASE}/api/pull          — pull a model programmatically")

## 10. Key Takeaways + Exercises

### What you learned this week

1. **Local LLMs via Ollama** give you privacy, zero cost, and offline capability — critical for pension data
2. **Messages have roles**: system (persona/constraints), user (input), assistant (history)
3. **Streaming** makes applications feel responsive — always use it in production
4. **Temperature** controls creativity vs determinism — use low temps for factual pension work
5. **System prompts** dramatically change model behavior without changing the underlying weights
6. **Context windows** are finite — 1000 articles overflow even generous limits — motivating RAG
7. **The `ollama` library** and **raw HTTP** are equivalent — understand both

### Exercises

**Exercise 1: Model comparison**
Run these 3 pension questions on both `llama3.2` and `mistral`, note which gives better answers:
- "What is the Policy Ladder (beleidsdekkingsgraad) and why does it matter?"
- "Explain the difference between defined benefit and defined contribution pension schemes."
- "What happens to a Dutch pension fund when the coverage ratio drops below 90%?"

Build a simple scoring rubric (1=wrong, 3=partially correct, 5=correct) and compare.

**Exercise 2: Multi-turn conversation loop**
Implement a simple REPL-style chat loop:
```python
history = [{"role": "system", "content": "You are a pension expert."}]
while True:
    user_input = input("You: ")
    if user_input == "quit": break
    history.append({"role": "user", "content": user_input})
    response = ollama.chat(model="llama3.2", messages=history)
    answer = response['message']['content']
    history.append({"role": "assistant", "content": answer})
    print(f"Bot: {answer}")
```
Try a conversation that builds on previous answers: first ask about coverage ratios, then "what if it drops below 105%?", then "and below 90%?"

**Exercise 3: Find the context limit**
Gradually increase the number of articles you stuff into the context. At what point does the model start giving worse answers? At what point does it return an error?